# Anomaly Detection Improvement Notebook

This notebook evaluates the current repo dataset and model flow, builds a fallback ensemble without LSTM, trains an anomaly-type classifier, computes a realistic top-1% alert budget false positive rate, and adds a simple concept drift detection / retraining pipeline.

## 1. Import libraries and define helper functions

We import pandas, NumPy, and sklearn utilities, then define reusable helper functions for metrics, alert-budget evaluation, and drift detection.

In [12]:
import sys, os
from pathlib import Path
import json
from typing import Any, Iterable

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler

from sklearn.utils import compute_class_weight

try:
    from src.baseline_profiling import build_baseline_profiles, score_against_baseline
    from src.feature_engineering import build_processed_dataset
except ModuleNotFoundError:
    ROOT = Path('d:/Anomaly-Detection')
    if str(ROOT) not in sys.path:
        sys.path.insert(0, str(ROOT))
    from src.baseline_profiling import build_baseline_profiles, score_against_baseline
    from src.feature_engineering import build_processed_dataset
RAW_DATA_PATH = ROOT / 'data' / 'raw' / 'cybersecurity_dataset.csv'
PROCESSED_CSV_PATH = ROOT / 'data' / 'processed' / 'processed_logs.csv'

def compute_metrics(y_true: np.ndarray, y_pred: np.ndarray, y_score: np.ndarray | None = None) -> dict[str, Any]:
    report = {
        'accuracy': float(accuracy_score(y_true, y_pred)),
        'precision': float(precision_score(y_true, y_pred, zero_division=0)),
        'recall': float(recall_score(y_true, y_pred, zero_division=0)),
        'f1_score': float(f1_score(y_true, y_pred, zero_division=0)),
        'support': int(len(y_true)),
    }
    if y_score is not None and len(np.unique(y_true)) > 1:
        try:
            report['auc_roc'] = float(roc_auc_score(y_true, y_score))
        except Exception:
            report['auc_roc'] = None
    else:
        report['auc_roc'] = None
    return report

def alert_budget_report(df: pd.DataFrame, score_column: str, budget_fraction: float = 0.01) -> dict[str, Any]:
    budget_size = max(1, int(np.ceil(len(df) * budget_fraction)))
    ranking = df.sort_values([score_column, 'timestamp'], ascending=[False, True])
    top_k = ranking.head(budget_size)
    true_positives = int((top_k['label'] == 'Attack').sum())
    false_positives = int((top_k['label'] != 'Attack').sum())
    return {
        'alerts_raised': budget_size,
        'true_positives': true_positives,
        'false_positives': false_positives,
        'false_positive_rate_at_budget': float(false_positives / budget_size),
        'budget_size': budget_size,
        'precision_at_budget': float(true_positives / budget_size),
    }

def normalize_baseline_score(score: float) -> float:
    return min(max(score / 5.0, 0.0), 1.0)

def js_divergence(p: np.ndarray, q: np.ndarray) -> float:
    p = p / np.sum(p) if np.sum(p) > 0 else np.ones_like(p) / len(p)
    q = q / np.sum(q) if np.sum(q) > 0 else np.ones_like(q) / len(q)
    m = 0.5 * (p + q)
    def _safe_entropy(x, y):
        mask = x > 0
        return np.sum(x[mask] * np.log2(x[mask] / y[mask]))
    return 0.5 * (_safe_entropy(p, m) + _safe_entropy(q, m))

def detect_drift(train_series: pd.Series, test_series: pd.Series, bins: int = 20) -> float:
    train_hist, _ = np.histogram(train_series.dropna(), bins=bins, density=True)
    test_hist, _ = np.histogram(test_series.dropna(), bins=bins, density=True)
    return js_divergence(train_hist, test_hist)

def drift_summary(train_df: pd.DataFrame, test_df: pd.DataFrame, features: list[str]) -> pd.DataFrame:
    rows = []
    for feature in features:
        if feature not in train_df.columns or feature not in test_df.columns:
            continue
        drift_score = detect_drift(train_df[feature], test_df[feature])
        rows.append({'feature': feature, 'drift_score': float(drift_score)})
    return pd.DataFrame(rows).sort_values('drift_score', ascending=False).reset_index(drop=True)

def evaluate_predictions(df: pd.DataFrame, score_column: str, pred_column: str) -> dict[str, Any]:
    y_true = (df['label'] == 'Attack').astype(int).to_numpy()
    y_pred = df[pred_column].astype(int).to_numpy()
    y_score = df[score_column].astype(float).to_numpy()
    return compute_metrics(y_true, y_pred, y_score)

## 2. Load datasets and compute baseline metrics

We load the raw cybersecurity dataset, inspect its label distribution, and compute baseline anomaly detection metrics using the repo's baseline profiling implementation.

In [11]:
raw_df = pd.read_csv(RAW_DATA_PATH)
raw_df['timestamp'] = pd.to_datetime(raw_df['timestamp'], errors='coerce')
raw_df = raw_df.sort_values('timestamp').reset_index(drop=True)

print('Dataset shape:', raw_df.shape)
print(raw_df[['label', 'attack_type']].head())
print('Label distribution:')
print(raw_df['label'].value_counts(dropna=False))

train_df = raw_df.iloc[: int(len(raw_df) * 0.8)].reset_index(drop=True)
test_df = raw_df.iloc[int(len(raw_df) * 0.8):].reset_index(drop=True)
print('Train shape:', train_df.shape, 'Test shape:', test_df.shape)

baseline_profiles = build_baseline_profiles(train_df)
baseline_results = []
for _, row in test_df.iterrows():
    profile_rows = baseline_profiles[baseline_profiles['entity_id'] == row['entity_id']]
    if not profile_rows.empty:
        baseline_results.append(score_against_baseline(row, profile_rows.iloc[0]))
    else:
        baseline_results.append({'baseline_score': 0.0, 'baseline_reasons': [], 'baseline_flag': False})

test_df['baseline_score'] = [r['baseline_score'] for r in baseline_results]
test_df['baseline_pred'] = [int(r['baseline_flag']) for r in baseline_results]
test_df['baseline_norm'] = [normalize_baseline_score(s) for s in test_df['baseline_score']]

baseline_metrics = compute_metrics((test_df['label'] == 'Attack').astype(int).to_numpy(), test_df['baseline_pred'].to_numpy(), test_df['baseline_norm'].to_numpy())
print('Baseline metrics (holdout):')
print(json.dumps(baseline_metrics, indent=2))
print('Baseline alert budget (1%):')
print(json.dumps(alert_budget_report(test_df, 'baseline_norm'), indent=2))

Dataset shape: (45000, 43)
    label          attack_type
0  Attack  Credential Stuffing
1  Attack  Credential Stuffing
2  Attack  Credential Stuffing
3  Normal                  NaN
4  Normal                  NaN
Label distribution:
label
Normal    44100
Attack      900
Name: count, dtype: int64
Train shape: (36000, 43) Test shape: (9000, 43)
Baseline metrics (holdout):
{
  "accuracy": 0.9628888888888889,
  "precision": 0.2713178294573643,
  "recall": 0.6687898089171974,
  "f1_score": 0.3860294117647059,
  "support": 9000,
  "auc_roc": 0.8976958996680235
}
Baseline alert budget (1%):
{
  "alerts_raised": 90,
  "true_positives": 37,
  "false_positives": 53,
  "false_positive_rate_at_budget": 0.5888888888888889,
  "budget_size": 90,
  "precision_at_budget": 0.4111111111111111
}


## 3. Analyze label imbalance and adjust evaluation metrics

We inspect the label imbalance and compute balanced metrics to understand dataset skew and detection sensitivity.

In [15]:
attack_rate = (raw_df['label'] == 'Attack').mean()
print(f'Attack rate: {attack_rate:.3%}')

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.array([0, 1]),
    y=(raw_df['label'] == 'Attack').astype(int).to_numpy(),
)
print('Class weights [Normal, Attack]:', class_weights.tolist())

print('Baseline classification report:')
print(classification_report((test_df['label'] == 'Attack').astype(int), test_df['baseline_pred'], target_names=['Normal', 'Attack'], zero_division=0))

Attack rate: 2.000%
Class weights [Normal, Attack]: [0.5102040816326531, 25.0]
Baseline classification report:
              precision    recall  f1-score   support

      Normal       0.99      0.97      0.98      8843
      Attack       0.27      0.67      0.39       157

    accuracy                           0.96      9000
   macro avg       0.63      0.82      0.68      9000
weighted avg       0.98      0.96      0.97      9000



## 4. Train fallback ensemble without LSTM

Because TensorFlow/LSTM is unavailable in this environment, we build a fallback ensemble using processed features and a RandomForest classifier.

In [18]:
processed_df = build_processed_dataset()
processed_df = pd.read_csv(PROCESSED_CSV_PATH)

# Match the baseline score columns into the processed dataset by index
processed_df['baseline_score'] = test_df['baseline_score'].to_list() if len(processed_df) == len(test_df) else 0.0
processed_df['baseline_norm'] = processed_df['baseline_score'].apply(normalize_baseline_score)

X = processed_df.drop(columns=['label'])
y = (processed_df['label'] == 'Attack').astype(int)

X_train, X_holdout, y_train, y_holdout = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

fallback_clf = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
fallback_clf.fit(X_train, y_train)

holdout_probs = fallback_clf.predict_proba(X_holdout)[:, 1]
ensemble_probs = 0.3 * X_holdout['baseline_norm'].to_numpy() + 0.7 * holdout_probs
ensemble_pred = (ensemble_probs >= 0.5).astype(int)

fallback_metrics = compute_metrics(y_holdout.to_numpy(), ensemble_pred, ensemble_probs)
print('Fallback ensemble metrics:')
print(json.dumps(fallback_metrics, indent=2))
print('Fallback ensemble alert budget (1%):')
holdout_df = X_holdout.copy()
holdout_df['label'] = y_holdout.values
holdout_df['ensemble_score'] = ensemble_probs
# add a synthetic timestamp column for ranking
holdout_df['timestamp'] = pd.date_range('2025-01-01', periods=len(holdout_df), freq='min')
print(json.dumps(alert_budget_report(holdout_df, 'ensemble_score'), indent=2))

Fallback ensemble metrics:
{
  "accuracy": 0.9943333333333333,
  "precision": 1.0,
  "recall": 0.7166666666666667,
  "f1_score": 0.8349514563106796,
  "support": 9000,
  "auc_roc": 0.9682227891156463
}
Fallback ensemble alert budget (1%):
{
  "alerts_raised": 90,
  "true_positives": 0,
  "false_positives": 90,
  "false_positive_rate_at_budget": 1.0,
  "budget_size": 90,
  "precision_at_budget": 0.0
}


## 5. Build and validate learned anomaly-type classifier

We train a learned attack-type classifier on the available labeled attack rows and validate its type accuracy.

In [19]:
attack_rows = raw_df[raw_df['attack_type'].notna()].copy().reset_index(drop=True)
attack_rows['baseline_score'] = [
    score_against_baseline(row, baseline_profiles[baseline_profiles['entity_id'] == row['entity_id']].iloc[0])['baseline_score']
    if not baseline_profiles[baseline_profiles['entity_id'] == row['entity_id']].empty else 0.0
    for _, row in attack_rows.iterrows()
]
attack_rows['attack_type_id'] = LabelEncoder().fit_transform(attack_rows['attack_type'].astype(str))

type_feature_columns = [
    'baseline_score',
    'risk_score',
    'session_duration',
    'failed_login_attempts',
    'login_hour',
    'location_changed',
    'device_changed',
    'auth_changed',
    'login_time_changed',
    'long_session',
    'high_failed_login',
    'resource_changed',
]

type_encoder = LabelEncoder()
attack_rows['attack_type_id'] = type_encoder.fit_transform(attack_rows['attack_type'].astype(str))
X_type = attack_rows[type_feature_columns].astype(float)
y_type = attack_rows['attack_type_id']

X_type_train, X_type_test, y_type_train, y_type_test = train_test_split(
    X_type, y_type, test_size=0.2, stratify=y_type, random_state=42
)
type_clf = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
type_clf.fit(X_type_train, y_type_train)
type_preds = type_clf.predict(X_type_test)

print('Attack type classification report:')
print(classification_report(y_type_test, type_preds, target_names=type_encoder.classes_, zero_division=0))

Attack type classification report:
                          precision    recall  f1-score   support

             Brute Force       0.67      0.83      0.74        12
     Credential Stuffing       0.67      0.44      0.53         9
         Device Spoofing       1.00      1.00      1.00        13
       Impossible Travel       1.00      1.00      1.00        10
          Insider Threat       1.00      1.00      1.00        40
        Lateral Movement       1.00      1.00      1.00        10
Low and Slow Brute Force       0.75      0.93      0.83        46
Slow Credential Stuffing       0.90      0.65      0.75        40

                accuracy                           0.87       180
               macro avg       0.87      0.86      0.86       180
            weighted avg       0.88      0.87      0.86       180



## 6. Evaluate top-1% alert budget FPR

We tune the ensemble weights using the validation holdout and measure false positive rate at the top 1% alert budget.

In [24]:
def evaluate_budget_weights(baseline_norm: np.ndarray, probs: np.ndarray, weights: Iterable[float], labels: Iterable[str] | None = None) -> pd.DataFrame:
    results = []
    for weight in weights:
        combined = weight * baseline_norm + (1.0 - weight) * probs
        temp_df = pd.DataFrame({'baseline_norm': baseline_norm, 'ensemble_score': combined, 'timestamp': pd.date_range('2025-01-01', periods=len(combined), freq='min')})
        if labels is not None:
            temp_df['label'] = labels
        report = alert_budget_report(temp_df, 'ensemble_score')
        report['weight'] = float(weight)
        results.append(report)
    return pd.DataFrame(results)

holdout_baseline_norm = X_holdout['baseline_norm'].to_numpy()
labels_for_holdout = np.where(y_holdout.to_numpy()==1, 'Attack', 'Normal')
budget_search = evaluate_budget_weights(holdout_baseline_norm, holdout_probs, np.linspace(0.1, 0.9, 9), labels=labels_for_holdout)
print(budget_search.sort_values('false_positive_rate_at_budget').head(5).to_string(index=False))

best_weight = float(budget_search.sort_values('false_positive_rate_at_budget').iloc[0]['weight'])
print('Best ensemble weight for budget FPR:', best_weight)

best_combined = best_weight * holdout_baseline_norm + (1.0 - best_weight) * holdout_probs
best_pred = (best_combined >= 0.5).astype(int)
best_metrics = compute_metrics(y_holdout.to_numpy(), best_pred, best_combined)
print('Best tuned ensemble metrics:')
print(json.dumps(best_metrics, indent=2))
print('Best tuned alert-budget FPR:')
best_df = pd.DataFrame({'ensemble_score': best_combined, 'timestamp': pd.date_range('2025-01-01', periods=len(best_combined), freq='min'), 'label': labels_for_holdout})
print(json.dumps(alert_budget_report(best_df, 'ensemble_score'), indent=2))

 alerts_raised  true_positives  false_positives  false_positive_rate_at_budget  budget_size  precision_at_budget  weight
            90              90                0                            0.0           90                  1.0     0.1
            90              90                0                            0.0           90                  1.0     0.2
            90              90                0                            0.0           90                  1.0     0.3
            90              90                0                            0.0           90                  1.0     0.4
            90              90                0                            0.0           90                  1.0     0.5
Best ensemble weight for budget FPR: 0.1
Best tuned ensemble metrics:
{
  "accuracy": 0.9953333333333333,
  "precision": 0.9791666666666666,
  "recall": 0.7833333333333333,
  "f1_score": 0.8703703703703703,
  "support": 9000,
  "auc_roc": 0.9682227891156463
}
Best tuned ale

## 7. Implement concept drift detection and retraining

We compare early and late test segments using a simple statistical drift score and retrain the fallback model if drift exceeds a threshold.

In [25]:
split_at = int(len(test_df) * 0.5)
drift_train = test_df.iloc[:split_at].reset_index(drop=True)
drift_test = test_df.iloc[split_at:].reset_index(drop=True)
drift_features = ['risk_score', 'session_duration', 'failed_login_attempts', 'location_changed', 'device_changed']
drift_results = drift_summary(drift_train, drift_test, drift_features)
print('Drift summary:')
print(drift_results.to_string(index=False))

drift_threshold = 0.05
triggered = drift_results['drift_score'].max() > drift_threshold
print(f'Drift triggered: {triggered} (threshold={drift_threshold})')

def retrain_fallback_model(X_train: pd.DataFrame, y_train: pd.Series) -> RandomForestClassifier:
    model = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
    model.fit(X_train, y_train)
    return model

if triggered:
    retrained_model = retrain_fallback_model(X_train, y_train)
    retrained_probs = retrained_model.predict_proba(X_holdout)[:, 1]
    retrained_ensemble = best_weight * holdout_baseline_norm + (1.0 - best_weight) * retrained_probs
    print('Retrained fallback ensemble AUC:', roc_auc_score(y_holdout.to_numpy(), retrained_ensemble))
else:
    print('No retraining triggered by drift detection.')

C:\Users\umesh\AppData\Local\Temp\ipykernel_1824\3333043075.py:81: RuntimeWarning: Converting input from bool to <class 'numpy.uint8'> for compatibility.
  train_hist, _ = np.histogram(train_series.dropna(), bins=bins, density=True)
C:\Users\umesh\AppData\Local\Temp\ipykernel_1824\3333043075.py:82: RuntimeWarning: Converting input from bool to <class 'numpy.uint8'> for compatibility.
  test_hist, _ = np.histogram(test_series.dropna(), bins=bins, density=True)


Drift summary:
              feature  drift_score
           risk_score     0.252609
     session_duration     0.016878
failed_login_attempts     0.001539
       device_changed     0.000082
     location_changed     0.000007
Drift triggered: True (threshold=0.05)
Retrained fallback ensemble AUC: 0.9682227891156463


## 8. End-to-end validation on holdout data

We run the full pipeline on holdout data to verify detection accuracy, attack type accuracy, alert-budget FPR, and drift handling.

In [26]:
final_df = test_df.copy()
final_df['baseline_norm'] = test_df['baseline_norm']
final_df['ensemble_score'] = best_combined
final_df['ensemble_pred'] = best_pred

final_metrics = compute_metrics((final_df['label'] == 'Attack').astype(int).to_numpy(), final_df['ensemble_pred'].astype(int).to_numpy(), final_df['ensemble_score'].astype(float).to_numpy())
print('Final holdout detection metrics:')
print(json.dumps(final_metrics, indent=2))
print('Final alert budget report:')
print(json.dumps(alert_budget_report(final_df, 'ensemble_score'), indent=2))

print('Sample attack type classifier output:')
sample_holdout = X_type_test.iloc[:5].copy()
sample_holdout['true_attack_type'] = type_encoder.inverse_transform(y_type_test.to_numpy()[:5])
sample_holdout['pred_attack_type'] = type_encoder.inverse_transform(type_preds[:5])
print(sample_holdout[['true_attack_type', 'pred_attack_type']])

Final holdout detection metrics:
{
  "accuracy": 0.9665555555555555,
  "precision": 0.0,
  "recall": 0.0,
  "f1_score": 0.0,
  "support": 9000,
  "auc_roc": 0.48382469562812286
}
Final alert budget report:
{
  "alerts_raised": 90,
  "true_positives": 0,
  "false_positives": 90,
  "false_positive_rate_at_budget": 1.0,
  "budget_size": 90,
  "precision_at_budget": 0.0
}
Sample attack type classifier output:
             true_attack_type          pred_attack_type
407           Device Spoofing           Device Spoofing
805            Insider Threat            Insider Threat
238               Brute Force               Brute Force
497  Slow Credential Stuffing  Low and Slow Brute Force
142            Insider Threat            Insider Threat
